In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

RAW_PATH = "../data/heart.csv"
CLEAN_PATH = "../data/heart_cleaned.csv"
PLOTS_DIR = "../output/heart_plots"

os.makedirs(PLOTS_DIR, exist_ok=True)

In [8]:
 # LOAD & UNDERSTAND THE DATASET
df = pd.read_csv(RAW_PATH)
print("Initial shape:", df.shape)

Initial shape: (1025, 14)


In [9]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
None


In [10]:
print(df.describe())

               age          sex           cp     trestbps        chol  \
count  1025.000000  1025.000000  1025.000000  1025.000000  1025.00000   
mean     54.434146     0.695610     0.942439   131.611707   246.00000   
std       9.072290     0.460373     1.029641    17.516718    51.59251   
min      29.000000     0.000000     0.000000    94.000000   126.00000   
25%      48.000000     0.000000     0.000000   120.000000   211.00000   
50%      56.000000     1.000000     1.000000   130.000000   240.00000   
75%      61.000000     1.000000     2.000000   140.000000   275.00000   
max      77.000000     1.000000     3.000000   200.000000   564.00000   

               fbs      restecg      thalach        exang      oldpeak  \
count  1025.000000  1025.000000  1025.000000  1025.000000  1025.000000   
mean      0.149268     0.529756   149.114146     0.336585     1.071512   
std       0.356527     0.527878    23.005724     0.472772     1.175053   
min       0.000000     0.000000    71.000000  

In [11]:
# 2. HANDLE MISSING VALUES
print("\nMissing values per column:\n", df.isnull().sum())


Missing values per column:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


In [12]:
# 3. REMOVE DUPLICATE RECORDS
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows found: {n_dupes}")
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


Duplicate rows found: 723
Shape after removing duplicates: (302, 14)


In [13]:
# 4. OUTLIER DETECTION (IQR method) on continuous numeric columns
continuous_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
 
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr
 
outlier_report = {}
for col in continuous_cols:
    low, high = iqr_bounds(df[col])
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    outlier_report[col] = n_out
print("\nOutlier counts (IQR method):", outlier_report)
 
# Cap (winsorize) extreme outliers instead of dropping rows, to preserve data.
for col in continuous_cols:
    low, high = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=low, upper=high)


Outlier counts (IQR method): {'age': 0, 'trestbps': 9, 'chol': 5, 'thalach': 1, 'oldpeak': 5}


In [14]:
# 5. EXPLORATORY DATA ANALYSIS
# 5.1 Max heart rate distribution by target
plt.figure(figsize=(7, 5))
sns.histplot(data=df, x="thalach", hue="target", kde=True, palette=["blue", "orange"], element="step")
plt.title("Max Heart Rate Achieved by Target")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/thalach_distribution.png")
plt.close()

In [15]:
# 5.1 Correlation heatmap
plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/correlation_heatmap.png")
plt.close()

In [16]:
# 5.2 Age distribution (Histogram)
plt.figure(figsize=(7, 5))
sns.histplot(df["age"], kde=True, bins=20, color="lightblue")
plt.title("Age Distribution")
plt.xlabel("Age")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/age_distribution.png")
plt.close()

In [17]:
# 5.3 Boxplots of continuous features by target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, continuous_cols):
    sns.boxplot(x="target", y=col, data=df, ax=ax, palette=["lightgreen", "red"])
    ax.set_title(f"{col} vs target")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/boxplots_by_target.png")
plt.close()

In [18]:
# 5.4 Target class distribution (Pie chart)
plt.figure(figsize=(5, 5))
df["target"].value_counts().plot.pie(
    labels=["Heart Disease (1)", "No Disease (0)"],
    autopct="%1.1f%%", colors=["green", "orange"], startangle=90
)
plt.ylabel("")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/target_pie_chart.png")
plt.close()
 

In [19]:
# 5.5 Chest pain type vs target (Bar chart)
plt.figure(figsize=(7, 5))
sns.countplot(x="cp", hue="target", data=df, palette=["lightblue", "red"])
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest Pain Type (cp)")
plt.legend(title="target", labels=["No Disease", "Disease"])
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/cp_vs_target_bar.png")
plt.close()

In [20]:
# 5.6 Sex vs target (Bar chart)
plt.figure(figsize=(6, 5))
sns.countplot(x="sex", hue="target", data=df, palette=["lightgreen", "orange"])
plt.title("Sex vs Heart Disease (0=Female, 1=Male)")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/sex_vs_target_bar.png")
plt.close()

In [21]:
# 6. FEATURE ENGINEERING: ENCODING CATEGORICAL VARIABLES
nominal_cols = ["cp", "restecg", "slope", "thal"]
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
print("\nShape after one-hot encoding:", df_encoded.shape)
print("Columns:", list(df_encoded.columns))


Shape after one-hot encoding: (302, 20)
Columns: ['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca', 'target', 'cp_1', 'cp_2', 'cp_3', 'restecg_1', 'restecg_2', 'slope_1', 'slope_2', 'thal_1', 'thal_2', 'thal_3']


In [22]:
# 7. FEATURE SCALING (StandardScaler on continuous numeric columns)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_encoded[continuous_cols] = scaler.fit_transform(df_encoded[continuous_cols])

In [23]:
# 8. SAVE CLEANED / PREPROCESSED DATASET
df_encoded.to_csv(CLEAN_PATH, index=False)
print(f"\nCleaned & preprocessed dataset saved to {CLEAN_PATH}")
print("Final shape:", df_encoded.shape)


Cleaned & preprocessed dataset saved to ../data/heart_cleaned.csv
Final shape: (302, 20)
